# 💰 QuickLoan — The Great Model Bake-Off
**Ensemble learning challenge · Steps 1–5 + EDA**

QuickLoan has to decide whether to approve or reject each applicant. The two kinds of mistake cost very different amounts:

| | Approve (predict 0) | Reject (predict 1) |
|---|---|---|
| **Repays** | +20% of loan (interest) | 0 (interest lost) |
| **Defaults** | −100% of loan 💸 | 0 (disaster avoided) |

So models are ranked on **money saved compared with approving everyone**, not on accuracy.

**Notebook plan:** 0 · Setup → EDA → 1 · Load & split → 2 · Weak single models → 3 · Four ensembles → 3b · Tuning for recall → 4 · Leaderboard → 5 · Verdict → Save model for `app.py`

## 0 · Setup

In [ ]:
# Colab already has XGBoost installed. On other machines, the cell below installs it if it's missing.
try:
    import xgboost
except ImportError:
    %pip install -q xgboost

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from sklearn.base import clone

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (VotingClassifier, BaggingClassifier, RandomForestClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier,
                              HistGradientBoostingClassifier)
from sklearn.model_selection import FixedThresholdClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:          # the brief allows this substitute
    HAS_XGB = False
print("XGBoost available:", HAS_XGB)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 1   # the same seed app.py uses

In [ ]:
# Load the data. In Colab, upload loans.csv through the Files panel (📁) or with the picker below.
if not os.path.exists("loans.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put loans.csv in the same folder as this notebook.")

df = pd.read_csv("loans.csv")
print(df.shape)
df.head()

## EDA — Exploratory Data Analysis
Before building any model, we look at what we're predicting, how unbalanced the classes are, which features separate defaulters from non-defaulters, and how much money is at stake.

In [ ]:
print(df.dtypes, "\n")
print("Missing values per column:\n", df.isna().sum(), "\n")
print("Duplicate rows:", df.duplicated().sum())
df.describe().T.round(2)

In [ ]:
# Target balance. This is the main reason accuracy misleads here.
rate = df["default"].mean()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df["default"].value_counts().sort_index().plot.bar(ax=ax[0], color=["#4C72B0", "#C44E52"])
ax[0].set_xticklabels(["Repaid (0)", "Defaulted (1)"], rotation=0); ax[0].set_title("Class balance")
for p in ax[0].patches: ax[0].annotate(f"{int(p.get_height()):,}", (p.get_x()+p.get_width()/2, p.get_height()), ha="center", va="bottom")

money = df.groupby("default")["loan_amount"].sum() / 1e6
money.plot.bar(ax=ax[1], color=["#4C72B0", "#C44E52"])
ax[1].set_xticklabels(["Repaid", "Defaulted"], rotation=0); ax[1].set_title("Total loan value (AED millions)")
plt.tight_layout(); plt.show()

print(f"Default rate: {rate:.1%}  ->  a model that predicts 'never defaults' already scores {1-rate:.1%} accuracy while catching 0 defaulters.")
print(f"Principal lent to defaulters across the dataset: AED {df.loc[df.default==1,'loan_amount'].sum():,.0f}")

In [ ]:
# How each numeric feature differs between defaulters and non-defaulters
num_cols = ["age","annual_income","loan_amount","loan_term_months","employment_years",
            "credit_history_months","num_existing_loans","past_defaults","debt_to_income"]
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.ravel(), num_cols):
    sns.kdeplot(data=df, x=col, hue="default", common_norm=False, fill=True, alpha=.35,
                palette={0:"#4C72B0",1:"#C44E52"}, ax=ax, warn_singular=False)
    ax.set_title(col)
plt.suptitle("Feature distributions: repaid (0) vs defaulted (1)", y=1.01, fontsize=14)
plt.tight_layout(); plt.show()

df.groupby("default")[num_cols].median().T.rename(columns={0:"median repaid",1:"median defaulted"}).round(2)

In [ ]:
# Default rate by the discrete or categorical drivers
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["past_defaults","num_existing_loans","home_ownership","loan_term_months"]):
    g = df.groupby(col)["default"].agg(["mean","size"])
    ax.bar(g.index.astype(str), g["mean"], color="#C44E52")
    ax.axhline(rate, ls="--", c="grey", label="overall rate")
    ax.set_title(f"Default rate by {col}"); ax.set_ylabel("default rate")
    for i,(m,n) in enumerate(zip(g["mean"], g["size"])): ax.text(i, m, f"n={n}", ha="center", va="bottom", fontsize=8)
axes[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
# Default rate across debt_to_income and credit-history bands
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["debt_to_income","credit_history_months"]):
    band = pd.qcut(df[col], 8, duplicates="drop")
    df.groupby(band, observed=True)["default"].mean().plot.bar(ax=ax, color="#DD8452")
    ax.axhline(rate, ls="--", c="grey"); ax.set_title(f"Default rate by {col} (octiles)")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Correlations, including with the target
corr = pd.get_dummies(df, columns=["home_ownership"], dtype=int).corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, annot_kws={"size":7})
plt.title("Correlation matrix"); plt.show()
corr["default"].drop("default").sort_values(key=abs, ascending=False).round(3).to_frame("corr with default")

### EDA takeaways
- **The data is clean.** It has no missing values and only one categorical column (`home_ownership`).
- **The classes are unbalanced.** About 1 in 8 applicants defaults, so a "do nothing" model already gets about 87% accuracy and catches no defaulters. This is why we track **recall** and **money saved**.
- The plots above show which features separate defaulters most clearly (see the correlation ranking and the default-rate bars). The usual credit-risk drivers are **past defaults, debt-to-income, existing loans and credit history**.
- Defaulters' loans are large in absolute terms. Each missed defaulter costs the **whole principal**, while a wrongly rejected good borrower costs only 20% of the loan. Each missed defaulter is therefore about **5× as expensive** as one wrongly rejected good borrower.

## Step 1 · Load, encode, split
- Encode with `pd.get_dummies(df, columns=["home_ownership"])`, **exactly as `app.py` does**, so the saved column list matches the dashboard.
- Split with **`test_size=0.25, random_state=1, stratify=y`**, the same split `app.py` uses, so the dashboard's money figure matches this leaderboard.
- Keep `loan_amount` for the test rows for the money metric.
- No leakage: every scaler, model and tuned threshold is fitted on the training set only.

In [ ]:
X = pd.get_dummies(df.drop(columns="default"), columns=["home_ownership"])
y = df["default"]

X_train, X_test, y_train, y_test, loan_train, loan_test = train_test_split(
    X, y, df["loan_amount"], test_size=0.25, random_state=RANDOM_STATE, stratify=y)

print("Columns:", list(X.columns))
print(f"Train: {X_train.shape}  default rate {y_train.mean():.3f}")
print(f"Test : {X_test.shape}  default rate {y_test.mean():.3f}  | defaulters in test: {y_test.sum()}")
print(f"Total principal of test-set defaulters (max achievable saving): AED {loan_test[y_test==1].sum():,.0f}")

In [ ]:
# The money metric exactly as given in the brief
def money_saved(y_true, y_pred, loan_amounts, interest=0.20):
    y_true = np.array(y_true); y_pred = np.array(y_pred); loan = np.array(loan_amounts)
    caught_defaulter = (y_pred==1) & (y_true==1)   # correctly rejected a defaulter -> loan SAVED
    wrongly_rejected = (y_pred==1) & (y_true==0)   # rejected a good borrower   -> interest LOST
    return loan[caught_defaulter].sum()  -  interest * loan[wrongly_rejected].sum()

MAX_SAVING = loan_test[y_test==1].sum()
print("Sanity checks (test set):")
print(f"  approve everyone : AED {money_saved(y_test, np.zeros(len(y_test)), loan_test):>14,.0f}")
print(f"  reject everyone  : AED {money_saved(y_test, np.ones(len(y_test)),  loan_test):>14,.0f}")
print(f"  perfect model    : AED {money_saved(y_test, y_test, loan_test):>14,.0f}")

In [ ]:
# Leaderboard helper. Every model is scored on the same test set.
results, fitted = [], {}

def evaluate(name, model, family, X_te=X_test, threshold=None, store=True):
    if threshold is None:
        pred = model.predict(X_te)
    else:
        pred = (model.predict_proba(X_te)[:, 1] >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    row = dict(Model=name, Family=family,
               Accuracy=accuracy_score(y_test, pred),
               Recall=recall_score(y_test, pred),
               Precision=precision_score(y_test, pred, zero_division=0),
               Caught=f"{tp}/{tp+fn}", Wrongly_rejected=fp,
               Money_saved=money_saved(y_test, pred, loan_test),
               Threshold=0.5 if threshold is None else round(threshold, 3))
    row["%_of_max"] = row["Money_saved"] / MAX_SAVING
    if store:
        results[:] = [r for r in results if r["Model"] != name] + [row]
        fitted[name] = model
    return row

def show(rows):
    return (pd.DataFrame(rows).style
            .format({"Accuracy":"{:.3f}","Recall":"{:.3f}","Precision":"{:.3f}",
                     "Money_saved":"AED {:,.0f}","%_of_max":"{:.1%}"}))

## Step 2 · Feel the pain: weak single models
These are three simple learners using default settings and the standard 0.5 cut-off. LR and KNN are wrapped with a `StandardScaler` in a pipeline because both are sensitive to feature scale. The pipeline also means the saved model includes its own scaler.

In [ ]:
singles = {
    "Decision Tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "Logistic Regression":     make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "KNN (k=15)":              make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
}
rows = []
for name, m in singles.items():
    m.fit(X_train, y_train)
    rows.append(evaluate(name, m, "Single"))
show(rows)

**Observation:** accuracy looks respectable (it's hard to fall much below the ~87% baseline), but **recall is low**. These models let many defaulters through, and each one costs the whole loan.

## Step 3 · The four ensembles
### 3.1 Voting: a committee of *different* models
Soft voting averages the predicted probabilities of the three single models above.

In [ ]:
voting = VotingClassifier(
    estimators=[("dt", clone(singles["Decision Tree (depth 3)"])),
                ("lr", clone(singles["Logistic Regression"])),
                ("knn", clone(singles["KNN (k=15)"]))],
    voting="soft")
voting.fit(X_train, y_train)
show([evaluate("Voting (soft: DT+LR+KNN)", voting, "Voting")])

### 3.2 Bagging: many copies of the *same* model on bootstrap samples (reduces **variance**)

In [ ]:
bagging = BaggingClassifier(DecisionTreeClassifier(random_state=RANDOM_STATE),
                            n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)
bagging.fit(X_train, y_train); rf.fit(X_train, y_train)
show([evaluate("Bagging (200 trees)", bagging, "Bagging"),
      evaluate("Random Forest", rf, "Bagging")])

### 3.3 Boosting: models in sequence, each fixing the last one's errors (reduces **bias**)

In [ ]:
ada = AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE)
gb  = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE)
if HAS_XGB:
    xgb = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.9,
                        colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    XGB_NAME = "XGBoost"
else:
    xgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=RANDOM_STATE)
    XGB_NAME = "HistGradientBoosting (XGB substitute)"
for m in (ada, gb, xgb): m.fit(X_train, y_train)
show([evaluate("AdaBoost", ada, "Boosting"),
      evaluate("Gradient Boosting", gb, "Boosting"),
      evaluate(XGB_NAME, xgb, "Boosting")])

### 3.4 Stacking: a "manager" model learns how much to trust each strong base model
The base models are RF, Gradient Boosting, XGBoost and scaled LR. The meta-model is Logistic Regression, trained on 5-fold out-of-fold predictions so it never sees leaked labels.

In [ ]:
stack = StackingClassifier(
    estimators=[("rf", clone(rf)), ("gb", clone(gb)), ("xgb", clone(xgb)),
                ("lr", clone(singles["Logistic Regression"]))],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    stack_method="predict_proba", n_jobs=-1)
stack.fit(X_train, y_train)
show([evaluate("Stacking (RF+GB+XGB+LR → LR)", stack, "Stacking")])

## Step 3b · Tuning for recall: making the models money-aware
Every model above used the default rule "reject if P(default) ≥ 0.5". That rule treats both mistakes as equally costly, which they aren't. We try three fixes:

1. **`class_weight='balanced'`** in LR and RF makes each defaulter count about 7× more in training.
2. **XGBoost `scale_pos_weight`** = (# repaid / # defaulted) does the same for boosting.
3. **Lower the decision threshold.** By expected value, reject when
   $p \cdot L > 0.20\,(1-p)\,L \;\Rightarrow\; p > \tfrac{0.20}{1.20} \approx 0.167$.
   Instead of trusting each model's probability calibration, we **tune the threshold with 5-fold cross-validation on the training set only**. We pick the cut-off that maximises money saved on out-of-fold predictions. The test set is never used for tuning.

In [ ]:
spw = (y_train==0).sum() / (y_train==1).sum()
print(f"scale_pos_weight = {spw:.2f}")

lr_bal = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
rf_bal = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
                                n_jobs=-1, random_state=RANDOM_STATE)
if HAS_XGB:
    xgb_w = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.9,
                          scale_pos_weight=spw, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    XGBW_NAME = "XGBoost (scale_pos_weight)"
else:
    xgb_w = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, class_weight="balanced",
                                           random_state=RANDOM_STATE)
    XGBW_NAME = "HistGB (balanced, XGB substitute)"
for m in (lr_bal, rf_bal, xgb_w): m.fit(X_train, y_train)
show([evaluate("Logistic Regression (balanced)", lr_bal, "Recall-tuned"),
      evaluate("Random Forest (balanced)", rf_bal, "Recall-tuned"),
      evaluate(XGBW_NAME, xgb_w, "Recall-tuned")])

In [ ]:
# Threshold tuning: choose the cut-off that maximises money on OUT-OF-FOLD training predictions
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
GRID = np.round(np.arange(0.05, 0.71, 0.01), 2)

def tune_threshold(model):
    oof = cross_val_predict(clone(model), X_train, y_train, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
    gains = [money_saved(y_train, (oof >= t).astype(int), loan_train) for t in GRID]
    return GRID[int(np.argmax(gains))], gains

to_tune = {
    "Logistic Regression":            singles["Logistic Regression"],
    "Random Forest":                  rf,
    "Gradient Boosting":              gb,
    XGB_NAME:                         xgb,
    "Voting (soft: DT+LR+KNN)":       voting,
    "Stacking (RF+GB+XGB+LR → LR)":   stack,
}
tuned_thr, curves, rows = {}, {}, []
for name, m in to_tune.items():
    t, gains = tune_threshold(m)
    tuned_thr[name], curves[name] = t, gains
    rows.append(evaluate(f"{name} @ tuned thr", m, "Threshold-tuned", threshold=t))
show(rows)

In [ ]:
# How money on the training folds changes with the threshold
plt.figure(figsize=(10, 5))
for name, g in curves.items():
    plt.plot(GRID, np.array(g)/1e6, label=f"{name} (best {tuned_thr[name]:.2f})")
plt.axvline(0.5, c="grey", ls="--"); plt.text(0.505, plt.ylim()[0], " default 0.5", color="grey")
plt.axvline(1/6, c="black", ls=":"); plt.text(1/6+0.005, plt.ylim()[0], " theory 0.167")
plt.xlabel("Decision threshold on P(default)"); plt.ylabel("Money saved on train OOF (AED m)")
plt.title("Money saved vs threshold (cross-validated on training data)"); plt.legend(fontsize=8); plt.show()

## Step 4 · 🏆 The leaderboard
Every model is scored on the **same 1,500-row test set** and ranked by money saved.

In [ ]:
lb = (pd.DataFrame(results).sort_values("Money_saved", ascending=False).reset_index(drop=True))
lb.index += 1
(lb.style
   .format({"Accuracy":"{:.3f}","Recall":"{:.3f}","Precision":"{:.3f}",
            "Money_saved":"AED {:,.0f}","%_of_max":"{:.1%}"})
   .background_gradient(subset=["Money_saved"], cmap="Greens")
   .background_gradient(subset=["Recall"], cmap="Blues")
   .background_gradient(subset=["Accuracy"], cmap="Greys"))

In [ ]:
order = lb.sort_values("Money_saved")
fam_col = {"Single":"#8C8C8C","Voting":"#937860","Bagging":"#55A868","Boosting":"#4C72B0",
           "Stacking":"#8172B3","Recall-tuned":"#DD8452","Threshold-tuned":"#C44E52"}
cols = order["Family"].map(fam_col)
fig, axes = plt.subplots(1, 3, figsize=(20, 9), sharey=True)
for ax, metric, fmt in zip(axes, ["Accuracy","Recall","Money_saved"], ["{:.3f}","{:.3f}","AED {:,.0f}"]):
    vals = order[metric]/1e6 if metric=="Money_saved" else order[metric]
    ax.barh(order["Model"], vals, color=cols)
    ax.set_title(metric.replace("_"," ") + (" (AED millions)" if metric=="Money_saved" else ""), fontsize=13)
    for i,v in enumerate(order[metric]): ax.text(vals.iloc[i], i, " "+fmt.format(v), va="center", fontsize=8)
    if metric=="Accuracy": ax.set_xlim(order[metric].min()-0.05, 1)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=c,label=f) for f,c in fam_col.items()], loc="lower center", ncol=7)
plt.suptitle("QuickLoan bake-off: same test set, three scores", fontsize=15)
plt.tight_layout(rect=[0,0.04,1,0.97]); plt.show()

In [ ]:
acc_range = lb["Accuracy"].max() - lb["Accuracy"].min()
money_range = lb["Money_saved"].max() - lb["Money_saved"].min()
print(f"Accuracy range across all models : {lb['Accuracy'].min():.3f} – {lb['Accuracy'].max():.3f}  ({acc_range*100:.1f} percentage points)")
print(f"Money-saved range across all models: AED {lb['Money_saved'].min():,.0f} – AED {lb['Money_saved'].max():,.0f}  (spread AED {money_range:,.0f})")
best_acc = lb.loc[lb['Accuracy'].idxmax()]; best_money = lb.iloc[0]
print(f"\nMost ACCURATE model : {best_acc['Model']}  -> acc {best_acc['Accuracy']:.3f}, saves AED {best_acc['Money_saved']:,.0f}")
print(f"Most PROFITABLE model: {best_money['Model']}  -> acc {best_money['Accuracy']:.3f}, saves AED {best_money['Money_saved']:,.0f}")

In [ ]:
# Confusion matrices: the most accurate model vs the most profitable model
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, r in zip(axes, [best_acc, best_money]):
    m = fitted[r["Model"]]
    pred = m.predict(X_test) if r["Threshold"]==0.5 else (m.predict_proba(X_test)[:,1] >= r["Threshold"]).astype(int)
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Approve","Reject"], yticklabels=["Repaid","Defaulted"])
    ax.set_title(f"{r['Model']}\nacc {r['Accuracy']:.3f} · saves AED {r['Money_saved']:,.0f}", fontsize=10)
    ax.set_xlabel("Model decision"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

## Step 5 · The verdict

In [ ]:
champ = lb.iloc[0]
print("RECOMMENDATION TO QUICKLOAN LEADERSHIP")
print("="*60)
print(f"Deploy            : {champ['Model']}")
print(f"Decision rule     : reject if P(default) >= {champ['Threshold']}")
print(f"Money saved       : AED {champ['Money_saved']:,.0f} vs approving everyone "
      f"({champ['%_of_max']:.1%} of the AED {MAX_SAVING:,.0f} theoretically recoverable)")
print(f"Defaulters caught : {champ['Caught']}  (recall {champ['Recall']:.1%})")
print(f"Good borrowers wrongly rejected: {champ['Wrongly_rejected']}")
print(f"Accuracy          : {champ['Accuracy']:.1%}")

### 📝 Recommendation to QuickLoan leadership

**Deploy the Stacking ensemble (Random Forest + Gradient Boosting + XGBoost + Logistic Regression, with a Logistic Regression manager on top), and reject any applicant whose predicted default probability is 14% or higher.**

| | Approve everyone (today) | Most *accurate* model (Random Forest) | **Recommended: Stacking @ 0.14** |
|---|---|---|---|
| Money saved on the 1,500 test loans | AED 0 | AED 4.61 m | **AED 6.11 m** |
| Defaulters caught | 0 / 188 | 60 / 188 | **102 / 188** |
| Good borrowers wrongly rejected | 0 | 26 | 131 |
| Accuracy | 87.5% | **89.7%** | 85.5% |

**Why this model:** on the same held-out test set it saves **AED 6.11 m compared with approving everyone**. That is about half (49.8%) of the AED 12.27 m that a perfect model could recover. It is **AED 1.5 m more** than the most accurate model, even though its accuracy is 4 points *lower*.

**The accuracy-vs-money insight:**
- Across all 19 models, accuracy only ranged from 76% to 90%, and every untuned model sat within about 2 points of the 87.5% you get by approving everyone. Money saved ranged from **AED 1.97 m to AED 6.11 m**, a 3x spread. Accuracy hides the difference that matters.
- The biggest single gain came from **changing the decision threshold, not from changing the algorithm**. The same Stacking model saves AED 4.74 m at the default 0.5 cut-off and AED 6.11 m at 0.14. Every model improved once its threshold was tuned.
- This matches the economics. Missing a defaulter costs the whole loan, while rejecting a good borrower costs only 20% interest, so it pays to reject once P(default) is above 0.2/1.2, which is about 0.17. The thresholds tuned by cross-validation (0.14 to 0.24) all landed near that value.
- Ensembles beat single models. Stacking and XGBoost were the top two, and at the 0.5 cut-off every ensemble except Voting and AdaBoost outperformed the best single model.

**Trade-offs we accept:**
1. **Lower accuracy and more rejected good customers.** About 131 of 1,312 good applicants (10%) get turned away. Their lost interest is already deducted from the AED 6.11 m.
2. **Less interpretability.** A stack of four models is hard to explain to a customer or a regulator. If explainability is required, the **threshold-tuned Logistic Regression** (reject at P ≥ 0.20) saves AED 5.87 m. That is only AED 0.24 m (4%) less, and its coefficients are fully interpretable. Running it alongside the stack as an explainable "challenger" is a sensible option.
3. **Small differences at the top may be noise.** The top five models are within about 5% of each other, based on only 188 test defaulters. Before rollout we should confirm the ranking on fresh data and monitor money saved monthly, re-tuning the threshold if interest rates or the default rate change.

*Figures are from the Colab run with XGBoost installed. Exact numbers can shift slightly with library versions.*

## Stretch prep · Save the champion for `app.py`
`app.py` calls `model.predict()`, so a tuned threshold must be **built into the saved model**. scikit-learn's `FixedThresholdClassifier` does this, and it can be saved and reloaded in the app without the notebook. The saved `columns` are the `get_dummies` columns, which `app.py` reindexes to.

In [ ]:
champ_name = champ["Model"]
base_name = champ_name.replace(" @ tuned thr", "")
base_model = fitted[champ_name]
if champ["Threshold"] != 0.5:
    deploy_model = FixedThresholdClassifier(clone(base_model), threshold=float(champ["Threshold"]),
                                            response_method="predict_proba").fit(X_train, y_train)
else:
    deploy_model = base_model

# Check: the deployed object reproduces the leaderboard's money figure on app.py's split
chk = money_saved(y_test, deploy_model.predict(X_test), loan_test)
print(f"Deployed model saves AED {chk:,.0f} on the test split (leaderboard: AED {champ['Money_saved']:,.0f})")

joblib.dump({"model": deploy_model, "columns": list(X_train.columns)}, "loan_model.joblib")
print("Saved loan_model.joblib ->", list(X_train.columns))

In [ ]:
# Optional: download the model in Colab so it can go next to app.py
try:
    from google.colab import files
    files.download("loan_model.joblib")
except ImportError:
    pass